# E-Commerce Analytics System
# Notebook 2: Data Cleaning & Validation

This notebook performs:
- Missing value handling
- Duplicate removal
- Data type correction
- Product normalization
- Email validation
- Referential integrity checks
- Data quality reporting
- Export of cleaned CSV files


In [ ]:
import pandas as pd
import re
from pathlib import Path

RAW=Path("data/raw")
CLEAN=Path("data/cleaned")
CLEAN.mkdir(parents=True,exist_ok=True)

customers=pd.read_csv(RAW/"customers.csv")
products=pd.read_csv(RAW/"products.csv")
orders=pd.read_csv(RAW/"orders.csv")
order_items=pd.read_csv(RAW/"order_items.csv")


## Helper Functions

In [ ]:
EMAIL_PATTERN=r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

def normalize_product_names(df):
    df["product_name"]=(
        df["product_name"]
        .astype(str)
        .str.strip()
        .str.title()
    )
    return df

def validate_emails(df):
    invalid=df.loc[
        ~df["email"].astype(str).str.match(EMAIL_PATTERN,na=False),
        ["customer_id","email"]
    ]
    return invalid

def parse_dates(series):
    dt=pd.to_datetime(series,errors="coerce")
    missing=dt.isna()
    if missing.any():
        dt.loc[missing]=pd.to_datetime(series[missing],format="%d-%m-%Y %H:%M:%S",errors="coerce")
    return dt

def quality_summary(name,df):
    print("="*60)
    print(name)
    print("Rows:",len(df))
    print("Duplicates:",df.duplicated().sum())
    print("Missing Values")
    print(df.isna().sum())


## Clean Customers

In [ ]:
customers=customers.drop_duplicates()

invalid_emails=validate_emails(customers)

customers["email"]=customers["email"].fillna("unknown@example.com")

customers["registration_date"]=pd.to_datetime(
    customers["registration_date"],
    errors="coerce"
)

quality_summary("CUSTOMERS",customers)
invalid_emails.head()


## Clean Products

In [ ]:
products=products.drop_duplicates()
products=normalize_product_names(products)

products["cost_price"]=pd.to_numeric(
    products["cost_price"],
    errors="coerce"
)

products["cost_price"]=products["cost_price"].fillna(
    products["cost_price"].median()
)

quality_summary("PRODUCTS",products)
products.head()


## Clean Orders

In [ ]:
orders=orders.drop_duplicates()

orders["order_date"]=parse_dates(orders["order_date"])

orders["customer_id"]=orders["customer_id"].fillna(-1).astype(int)

future_orders=orders[
    orders["order_date"]>pd.Timestamp.today()
]

quality_summary("ORDERS",orders)

future_orders.head()


## Clean Order Items

In [ ]:
order_items=order_items.drop_duplicates()

order_items["discount_percent"]=order_items["discount_percent"].clip(0,100)

order_items.loc[
    order_items["quantity"]<0,
    "return_flag"
]="Returned"

order_items.loc[
    order_items["quantity"]>=0,
    "return_flag"
]="Purchased"

quality_summary("ORDER ITEMS",order_items)

order_items.head()


## Referential Integrity

In [ ]:
invalid_order_refs=order_items[
    ~order_items["order_id"].isin(orders["order_id"])
]

invalid_product_refs=order_items[
    ~order_items["product_id"].isin(products["product_id"])
]

print("Invalid Order References :",len(invalid_order_refs))
print("Invalid Product References :",len(invalid_product_refs))

invalid_order_refs.head()


## Data Quality Report

In [ ]:
report={
"Invalid Emails":len(invalid_emails),
"Future Orders":len(future_orders),
"Broken Order FK":len(invalid_order_refs),
"Broken Product FK":len(invalid_product_refs),
"Duplicate Customers":customers.duplicated().sum(),
"Duplicate Products":products.duplicated().sum(),
"Duplicate Orders":orders.duplicated().sum(),
"Duplicate OrderItems":order_items.duplicated().sum()
}

quality_report=pd.DataFrame(
    report.items(),
    columns=["Issue","Count"]
)

quality_report


## Export Cleaned Data

In [ ]:
customers.to_csv(CLEAN/"customers_clean.csv",index=False)
products.to_csv(CLEAN/"products_clean.csv",index=False)
orders.to_csv(CLEAN/"orders_clean.csv",index=False)
order_items.to_csv(CLEAN/"order_items_clean.csv",index=False)
quality_report.to_csv(CLEAN/"data_quality_report.csv",index=False)

print("Clean datasets exported successfully.")
